# Part 3 — Recursive Function Simulation
**Algorithm Analysis and Simulation Toolkit | Term 2, SY 2025–2026**

---
## What is Recursion?
A function is **recursive** when it calls itself. Every recursive function needs:
1. **Base case** — the condition that stops the recursion
2. **Recursive case** — the call that reduces the problem toward the base case

---
## Algorithm Explanations

| Function | Base Case | Recursive Case | Complexity |
|---|---|---|---|
| Factorial(n) | n≤1 → return 1 | n × factorial(n-1) | O(n) calls, O(n) depth |
| Fibonacci(n) | n≤1 → return n | fib(n-1) + fib(n-2) | O(2ⁿ) calls naive, O(n) memoized |
| Tower of Hanoi(n) | n=1 → move disk | move n-1, move n, move n-1 | O(2ⁿ-1) moves |
| Binary Search(arr,t) | lo>hi → not found; arr[mid]==t → found | search left or right half | O(log n) depth |
| GCD(a,b) | b=0 → return a | gcd(b, a mod b) | O(log min(a,b)) |
| Power(base,exp) | exp=0 → return 1 | half²  or  base×power(base,exp-1) | O(log n) |

---
## Algorithm Flowcharts

### Factorial
```
factorial(n)
  └─► n ≤ 1? → return 1  (base case)
  └─► return n × factorial(n-1)  (recursive case)

Call tree for factorial(4):
  factorial(4)
    └─► 4 × factorial(3)
          └─► 3 × factorial(2)
                └─► 2 × factorial(1)
                      └─► return 1  ← base case
                return 2×1 = 2
          return 3×2 = 6
    return 4×6 = 24
```

### Tower of Hanoi
```
hanoi(n, src, aux, dst)
  └─► n == 1? → Move disk 1 from src to dst  (base case)
  └─► hanoi(n-1, src, dst, aux)   ← move top n-1 to aux
  └─► Move disk n from src to dst
  └─► hanoi(n-1, aux, src, dst)   ← move n-1 from aux to dst
```

---
## Output Format (matches spec exactly)
```
factorial(4)
-> 4 * factorial(3)
   -> 3 * factorial(2)
      -> 2 * factorial(1)
         -> base case: return 1
      -> 2 * factorial(1) = 2
   -> 3 * factorial(2) = 6
-> 4 * factorial(3) = 24

Result = 24
```

---
## Program Instructions
1. Run **Cell 2** to load all recursive functions.
2. Choose a function, set n, and optionally fill the 2nd param field.
3. Click **Run** to see the full call trace and result.
4. The **Profiler table** shows how call count and depth grow with n.
5. Run **Test Cases** to verify all functions are correct.


In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# shared indent helper — 2 spaces per depth level
def _i(d): return '  ' * d

# ── FACTORIAL ─────────────────────────────────────────────────────────────────
# Computes n! recursively. Base case: n≤1 returns 1.
# Output matches spec format: "-> n * factorial(n-1)"
def factorial(n, depth=0, log=None):
    if log is None:
        log = []
        if not isinstance(n, int) or n < 0:
            raise ValueError(f'factorial requires a non-negative integer, got {n!r}')
    log.append(f'{_i(depth)}factorial({n})')
    if n <= 1:
        log.append(f'{_i(depth)}-> base case: return 1')
        return 1, log
    sub, _ = factorial(n - 1, depth + 1, log)
    result = n * sub
    log.append(f'{_i(depth)}-> {n} * factorial({n-1}) = {result}')
    return result, log

# ── FIBONACCI ─────────────────────────────────────────────────────────────────
# Naive: recomputes the same values — O(2^n) calls.
# Memoized: stores results in a dict — O(n) calls. Dramatically faster.
def fibonacci(n, depth=0, log=None, memo=None, use_memo=False):
    if log  is None: log  = []
    if memo is None: memo = {}
    if not (isinstance(n, int) and n >= 0):
        raise ValueError(f'fibonacci requires a non-negative integer, got {n!r}')
    if use_memo and n in memo:
        log.append(f'{_i(depth)}fibonacci({n}) -> [cached] {memo[n]}')
        return memo[n], log
    log.append(f'{_i(depth)}fibonacci({n})')
    if n <= 1:
        log.append(f'{_i(depth)}-> base case: return {n}')
        if use_memo: memo[n] = n
        return n, log
    l, _ = fibonacci(n-1, depth+1, log, memo, use_memo)
    r, _ = fibonacci(n-2, depth+1, log, memo, use_memo)
    result = l + r
    if use_memo: memo[n] = result
    log.append(f'{_i(depth)}-> fibonacci({n-1}) + fibonacci({n-2}) = {l} + {r} = {result}')
    return result, log

# ── TOWER OF HANOI ────────────────────────────────────────────────────────────
# Moves n disks from src to dst using aux as a helper peg.
# Minimum moves = 2^n - 1. Output matches spec: "Move disk k from X to Y"
def tower_of_hanoi(n, src='A', aux='B', dst='C', depth=0, log=None, moves=None):
    if log   is None: log   = []
    if moves is None:
        moves = []
        if not (isinstance(n, int) and n >= 1):
            raise ValueError(f'n must be a positive integer, got {n!r}')
    log.append(f'{_i(depth)}hanoi({n}, {src}->{dst} via {aux})')
    if n == 1:
        m = f'Move disk 1 from {src} to {dst}'
        moves.append(m); log.append(f'{_i(depth)}-> {m}')
        return moves, log
    tower_of_hanoi(n-1, src, dst, aux, depth+1, log, moves)
    m = f'Move disk {n} from {src} to {dst}'
    moves.append(m); log.append(f'{_i(depth)}-> {m}')
    tower_of_hanoi(n-1, aux, src, dst, depth+1, log, moves)
    return moves, log

# ── BINARY SEARCH ─────────────────────────────────────────────────────────────
# Recursively halves the search space. Requires a sorted input array.
# Returns the index of the target or -1 if not found.
def binary_search(arr, target, lo=0, hi=None, depth=0, log=None):
    if log is None: log = []
    if hi  is None: hi  = len(arr) - 1
    log.append(f'{_i(depth)}binary_search(arr[{lo}..{hi}], target={target})')
    if lo > hi:
        log.append(f'{_i(depth)}-> not found'); return -1, log
    mid = (lo + hi) // 2
    log.append(f'{_i(depth)}-> mid={mid}, arr[mid]={arr[mid]}')
    if arr[mid] == target:
        log.append(f'{_i(depth)}-> FOUND at index {mid} ✓'); return mid, log
    if arr[mid] < target:
        return binary_search(arr, target, mid+1, hi, depth+1, log)
    return binary_search(arr, target, lo, mid-1, depth+1, log)

# ── GCD (Euclidean) ───────────────────────────────────────────────────────────
# Repeatedly applies: gcd(a, b) = gcd(b, a mod b) until b = 0.
def gcd(a, b, depth=0, log=None):
    if log is None:
        log = []
        if not (isinstance(a, int) and isinstance(b, int) and a >= 0 and b >= 0):
            raise ValueError('gcd requires non-negative integers.')
    log.append(f'{_i(depth)}gcd({a}, {b})')
    if b == 0:
        log.append(f'{_i(depth)}-> base case: return {a}'); return a, log
    result, _ = gcd(b, a % b, depth+1, log)
    log.append(f'{_i(depth)}-> gcd({b}, {a}%{b}={a%b}) = {result}')
    return result, log

# ── FAST POWER (exponentiation by squaring) ───────────────────────────────────
# Computes base^exp in O(log n) recursive calls by squaring half-results.
def power(base, exp, depth=0, log=None):
    if log is None:
        log = []
        if not (isinstance(exp, int) and exp >= 0):
            raise ValueError('exp must be a non-negative integer.')
    log.append(f'{_i(depth)}power({base}, {exp})')
    if exp == 0:
        log.append(f'{_i(depth)}-> base case: return 1'); return 1, log
    if exp % 2 == 0:
        half, _ = power(base, exp//2, depth+1, log)
        result = half * half
        log.append(f'{_i(depth)}-> (power({base},{exp//2}))^2 = {half}^2 = {result}')
    else:
        sub, _ = power(base, exp-1, depth+1, log)
        result = base * sub
        log.append(f'{_i(depth)}-> {base} * power({base},{exp-1}) = {base}*{sub} = {result}')
    return result, log

# ── MERGE SORT RECURSIVE TRACE ────────────────────────────────────────────────
# Shows the full divide-and-conquer decomposition of merge sort.
def merge_sort_trace(arr, depth=0, log=None):
    if log is None: log = []
    log.append(f'{_i(depth)}merge_sort({arr})')
    if len(arr) <= 1:
        log.append(f'{_i(depth)}-> base case: {arr}'); return arr, log
    mid = len(arr) // 2
    left,  _ = merge_sort_trace(arr[:mid], depth+1, log)
    right, _ = merge_sort_trace(arr[mid:], depth+1, log)
    merged = []; i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: merged.append(left[i]); i += 1
        else: merged.append(right[j]); j += 1
    merged.extend(left[i:] or right[j:])
    log.append(f'{_i(depth)}-> merge({left}, {right}) = {merged}')
    return merged, log

print('=== Part 3: Recursive Function Simulation ===')
print('  Functions loaded: factorial, fibonacci, tower_of_hanoi, binary_search, gcd, power, merge_sort_trace')
print('  Scroll down to run the simulator, profiler, and test cases.')


=== Part 3: Recursive Function Simulation ===
  Functions loaded: factorial, fibonacci, tower_of_hanoi, binary_search, gcd, power, merge_sort_trace
  Scroll down to run the simulator, profiler, and test cases.


## Interactive Simulator
Choose a function, set n, and optionally a 2nd parameter. Click Run.

In [2]:
# ── RECURSIVE FUNCTION SIMULATOR ─────────────────────────────────────────────
# Dropdown selects the function. Slider sets n.
# 2nd param field is used for: GCD's b value, binary search target, fast power base.

FN_LIST = [
    'Factorial',
    'Fibonacci (naive)',
    'Fibonacci (memoized)',
    'Tower of Hanoi',
    'Binary Search',
    'GCD (Euclidean)',
    'Fast Power (base^n)',
    'Merge Sort Trace',
]

w_fn  = widgets.Dropdown(
    options=FN_LIST, value='Factorial',
    description='Function:', layout=widgets.Layout(width='280px'),
    style={'description_width': 'initial'}
)
w_n   = widgets.IntSlider(
    value=5, min=1, max=15,
    description='n / disks:', style={'description_width': 'initial'}
)
w_b   = widgets.Text(
    value='',
    placeholder='GCD: b value  |  Binary Search: target  |  Power: base  |  Merge Sort: array (csv)',
    description='2nd param:', layout=widgets.Layout(width='500px'),
    style={'description_width': 'initial'}
)
w_run = widgets.Button(
    description='▶  Run',
    button_style='success', layout=widgets.Layout(width='100px', height='36px')
)
out_r = widgets.Output()

def run_sim(_):
    with out_r:
        clear_output(wait=True)
        fn = w_fn.value; n = w_n.value; braw = w_b.value.strip()
        try:
            if fn == 'Factorial':
                result, log = factorial(n)
                print(f'=== FACTORIAL({n}) ===\n')
                print('\n'.join(log))
                print(f'\nResult = {result}')

            elif fn == 'Fibonacci (naive)':
                cap = min(n, 8)
                if n > 8: print(f'(Trace capped at n=8 — naive fibonacci has 2^n calls)\n')
                result, log = fibonacci(cap, use_memo=False)
                print(f'=== FIBONACCI({cap}) — Naive Recursive ===\n')
                print('\n'.join(log))
                print(f'\nResult = {result}')

            elif fn == 'Fibonacci (memoized)':
                result, log = fibonacci(n, use_memo=True)
                print(f'=== FIBONACCI({n}) — Memoized ===\n')
                print('\n'.join(log))
                print(f'\nResult = {result}')
                print(f'(Notice: cached values avoid redundant calls)')

            elif fn == 'Tower of Hanoi':
                moves, log = tower_of_hanoi(n)
                print(f'=== TOWER OF HANOI ({n} disk{"s" if n>1 else ""}) ===\n')
                if n <= 4:
                    print('Call tree:')
                    print('\n'.join(log))
                    print()
                print('Move sequence:')
                for i, m in enumerate(moves, 1):
                    print(f'  {i:3d}. {m}')
                print(f'\nTotal moves : {len(moves)}')
                print(f'Formula 2^n-1 : {2**n - 1}')

            elif fn == 'Binary Search':
                arr = list(range(0, n * 2, 2))
                try: target = int(braw) if braw else arr[n // 2]
                except ValueError: print('⚠  2nd param should be the search target (integer).'); return
                result, log = binary_search(arr, target)
                print(f'=== BINARY SEARCH ===\n')
                print(f'Array  : {arr}')
                print(f'Target : {target}\n')
                print('\n'.join(log))
                print(f'\nResult = {"index " + str(result) if result >= 0 else "not found"}')

            elif fn == 'GCD (Euclidean)':
                try: b = int(braw) if braw else 18
                except ValueError: print('⚠  2nd param should be an integer (b value).'); return
                result, log = gcd(n, b)
                print(f'=== GCD({n}, {b}) ===\n')
                print('\n'.join(log))
                print(f'\nResult = {result}')

            elif fn == 'Fast Power (base^n)':
                try: base = int(braw) if braw else 2
                except ValueError: print('⚠  2nd param should be the base (integer).'); return
                result, log = power(base, n)
                print(f'=== FAST POWER: {base}^{n} ===\n')
                print('\n'.join(log))
                print(f'\nResult = {result}   (verify: {base}^{n} = {base**n})')

            elif fn == 'Merge Sort Trace':
                raw = braw if braw else '5, 2, 8, 1, 9, 3, 7, 4'
                try: arr = [int(x.strip()) for x in raw.split(',') if x.strip()]
                except ValueError: print('⚠  2nd param should be comma-separated integers.'); return
                if len(arr) > 10: arr = arr[:10]; print('⚠  Capped at 10 elements.\n')
                result, log = merge_sort_trace(arr)
                print(f'=== MERGE SORT RECURSIVE TRACE ===\n')
                print(f'Input  : {arr}\n')
                print('\n'.join(log))
                print(f'\nSorted : {result}')

        except Exception as e:
            print(f'⚠  Error: {e}')

w_run.on_click(run_sim)
display(widgets.VBox([widgets.HBox([w_fn, w_n]), w_b, w_run, out_r]))


## Call Count & Depth Profiler

In [4]:
# ── CALL COUNT & DEPTH PROFILER ───────────────────────────────────────────────
# Wraps plain recursive functions to count total calls and max depth.
# Runs across a range of n values to show growth rates.

class Profiler:
    def __init__(self, fn): self.fn=fn; self.calls=0; self.d=0; self.md=0
    def __call__(self, *a, **kw):
        self.calls+=1; self.d+=1; self.md=max(self.md, self.d)
        r = self.fn(*a, **kw); self.d-=1; return r
    def reset(self): self.calls=self.d=self.md=0

def _fact(n): return 1 if n<=1 else n*_fact(n-1)
def _fib(n):  return n if n<=1 else _fib(n-1)+_fib(n-2)
pf=Profiler(_fact); _fact=pf
pb=Profiler(_fib);  _fib=pb

ns = list(range(1, 14))
print('Call Count & Depth Growth — Factorial vs Fibonacci\n')
print(f"{'n':>3}  {'fact_calls':>12}  {'fact_depth':>11}  {'fib_calls':>11}  {'fib_depth':>10}")
print('─' * 56)
for n in ns:
    pf.reset(); pb.reset()
    _fact(n); _fib(n)
    print(f'{n:>3}  {pf.calls:>12}  {pf.md:>11}  {pb.calls:>11}  {pb.md:>10}')

print()
print('Observation: factorial is O(n) calls, fibonacci naive is O(2^n) calls.')
print('Memoization reduces fibonacci to O(n) calls — same as factorial.')


Call Count & Depth Growth — Factorial vs Fibonacci

  n    fact_calls   fact_depth    fib_calls   fib_depth
────────────────────────────────────────────────────────
  1             1            1            1           1
  2             2            2            3           2
  3             3            3            5           3
  4             4            4            9           4
  5             5            5           15           5
  6             6            6           25           6
  7             7            7           41           7
  8             8            8           67           8
  9             9            9          109           9
 10            10           10          177          10
 11            11           11          287          11
 12            12           12          465          12
 13            13           13          753          13

Observation: factorial is O(n) calls, fibonacci naive is O(2^n) calls.
Memoization reduces fibonacci to O(

## Automated Test Cases

In [5]:
# ── AUTOMATED TEST CASES ──────────────────────────────────────────────────────
# Verifies all recursive functions produce correct results.

import math

REC_TESTS = [
    ('factorial(0)',  lambda: factorial(0)[0],  1),
    ('factorial(1)',  lambda: factorial(1)[0],  1),
    ('factorial(5)',  lambda: factorial(5)[0],  120),
    ('factorial(10)', lambda: factorial(10)[0], 3628800),
    ('fibonacci(0)',  lambda: fibonacci(0)[0],  0),
    ('fibonacci(1)',  lambda: fibonacci(1)[0],  1),
    ('fibonacci(8)',  lambda: fibonacci(8)[0],  21),
    ('fibonacci(10, memoized)', lambda: fibonacci(10, use_memo=True)[0], 55),
    ('hanoi(1) moves',  lambda: len(tower_of_hanoi(1)[0]),  1),
    ('hanoi(3) moves',  lambda: len(tower_of_hanoi(3)[0]),  7),
    ('hanoi(5) moves',  lambda: len(tower_of_hanoi(5)[0]),  31),
    ('binary_search found',     lambda: binary_search([1,3,5,7,9,11], 7)[0],  3),
    ('binary_search not found', lambda: binary_search([1,3,5,7,9,11], 4)[0], -1),
    ('gcd(48, 18)',   lambda: gcd(48, 18)[0],  6),
    ('gcd(100, 75)',  lambda: gcd(100, 75)[0], 25),
    ('power(2, 10)',  lambda: power(2, 10)[0],  1024),
    ('power(3, 5)',   lambda: power(3, 5)[0],   243),
    ('power(x, 0)',   lambda: power(7, 0)[0],   1),
]

w_rt = widgets.Button(
    description='▶  Run Recursion Tests',
    button_style='info', layout=widgets.Layout(width='200px', height='36px')
)
out_rt = widgets.Output()

def run_rec_tests(_):
    with out_rt:
        clear_output(wait=True)
        print('=== Recursion Automated Test Cases ===\n')
        total = passed = 0
        for name, fn, expected in REC_TESTS:
            total += 1
            try:
                result = fn()
                ok = result == expected
                if ok: passed += 1
                sym = '✓' if ok else '✗'
                print(f'  {sym}  {name:<35}  expected={expected}, got={result}')
            except Exception as e:
                print(f'  ✗  {name:<35}  ERROR: {e}')
        print(f'\nResults: {passed}/{total} tests passed.')
        if passed == total: print('All recursive functions verified correct! ✓')

w_rt.on_click(run_rec_tests)
display(widgets.VBox([w_rt, out_rt]))
